In [1]:
# Check our GPU is working
import torch

print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU available: True
GPU name: Tesla T4
GPU memory: 15.6 GB


## 🏥 MedReport Explain — MedGemma Impact Challenge

> Fine-tuned MedGemma for multilingual medical report explanation.Supports Hindi, Spanish, Arabic, French, Bengali.


## Step 1: Install Libraries

In [2]:
%%capture
!pip install -q transformers accelerate peft trl bitsandbytes datasets --upgrade


## Step 2: Load Dataset

In [3]:
import pandas as pd
df = pd.read_csv('/kaggle/input/datasets/tboyle10/medicaltranscriptions/mtsamples.csv')

In [4]:
print(f'Total rows :{len(df)}')
print(f'columns:{df.columns.tolist()}')
df.sample(5)

Total rows :4999
columns:['Unnamed: 0', 'description', 'medical_specialty', 'sample_name', 'transcription', 'keywords']


,Unnamed: 0,description,medical_specialty,sample_name,transcription,keywords
3622,3622,The patient with a recent change in bowel fun...,Gastroenterology,Colonoscopy with Biopsy - 1,"PREPROCEDURE DIAGNOSIS:, Change in bowel func...","gastroenterology, change in bowel function, iv..."
3118,3118,Newly diagnosed T-cell lymphoma. The patient...,Hematology - Oncology,T-Cell Lymphoma Consult,"CHIEF COMPLAINT: , Newly diagnosed T-cell lymp...","hematology - oncology, t-cell lymphoma, subman..."
2915,2915,"CT head without contrast, CT facial bones wit...",Neurology,"CT Head, Facial Bones, Cervical Spine","EXAM: , CT head without contrast, CT facial bo...","neurology, sagittal, coronal, soft tissue swel..."
4554,4554,Blood in toilet. Questionable gastrointestin...,Consult - History and Phy.,Blood In Toilet,"CHIEF COMPLAINT: ,Blood in toilet.,HISTORY: ,...",NaN
1358,1358,History of right leg pain. Leg pain is no lo...,SOAP / Chart / Progress Notes,Leg Pain - Progress Note,"HISTORY OF PRESENT ILLNESS: , The patient is a...",NaN


In [5]:
print(df.info())
print(df.isnull().sum())
print(f'\ntotal rows before cleaning: {len(df)}')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4999 entries, 0 to 4998
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Unnamed: 0         4999 non-null   int64 
 1   description        4999 non-null   object
 2   medical_specialty  4999 non-null   object
 3   sample_name        4999 non-null   object
 4   transcription      4966 non-null   object
 5   keywords           3931 non-null   object
dtypes: int64(1), object(5)
memory usage: 234.5+ KB
None
Unnamed: 0              0
description             0
medical_specialty       0
sample_name             0
transcription          33
keywords             1068
dtype: int64

total rows before cleaning: 4999


In [6]:
df = df[['description','medical_specialty','transcription']].dropna()
df = df[df['transcription'].str.len()>100]
df = df.reset_index(drop=True)

print(f'rows after:{len(df)}')


rows after:4920


In [7]:
print(df['medical_specialty'].value_counts().head(10))

medical_specialty
Surgery                          1081
Consult - History and Phy.        511
Cardiovascular / Pulmonary        368
Orthopedic                        346
Radiology                         271
General Medicine                  257
Neurology                         223
Gastroenterology                  220
SOAP / Chart / Progress Notes     165
Obstetrics / Gynecology           155
Name: count, dtype: int64


## Step 3: Create Training Prompts

In [8]:
SPECIALTY_QUESTIONS = {
    'Surgery': [
        "What complications should I watch for after this procedure?",
        "How long is the recovery time and what restrictions do I have?",
        "When should I schedule my follow-up appointment?"
    ],
    'Cardiovascular / Pulmonary': [
        "What dietary and lifestyle changes should I make?",
        "What symptoms should make me call you or go to the ER immediately?",
        "Do I need to monitor my blood pressure or heart rate at home?"
    ],
    'Orthopedic': [
        "What activities should I avoid during recovery?",
        "Do I need physiotherapy and for how long?",
        "What level of pain is normal and when should I be concerned?"
    ],
    'Neurology': [
        "Are these symptoms permanent or can they improve?",
        "What activities or situations should I avoid?",
        "What warning signs should make me seek emergency care?"
    ],
    'Radiology': [
        "How serious are the findings in this imaging report?",
        "Do I need further imaging or tests?",
        "How urgently do I need to follow up on these findings?"
    ],
    'Allergy / Immunology': [
        "Should I get a formal allergy test to identify my specific triggers?",
        "Are there any foods, environments or activities I should avoid?",
        "Is this condition likely to get worse over time without treatment?"
    ],
    'General Medicine': [
        "Do I need to see a specialist based on these results?",
        "What lifestyle changes would help my condition?",
        "How often should I come back for monitoring?"
    ],
}

DEFAULT_QUESTIONS = [
    "What do these findings mean for my long-term health?",
    "What are the treatment options available for my specific situation?",
    "What warning signs should make me seek immediate medical care?"
]

def get_questions(specialty):
    for key in SPECIALTY_QUESTIONS:
        if key.lower() in specialty.lower():
            return SPECIALTY_QUESTIONS[key]
    return DEFAULT_QUESTIONS

def create_prompt_completion(transcription, description, specialty, language='English'):
    questions = get_questions(specialty)
    
    sentences = [s.strip() for s in transcription[:800].replace(',', '.').split('.')
                 if len(s.strip()) > 30][:5]
    
    findings = []
    for s in sentences[:3]:
        if any(h in s.upper() for h in ['SUBJECTIVE', 'OBJECTIVE', 'PLAN', 'ASSESSMENT']):
            continue
        findings.append(s)
    
    if len(findings) < 2:
        findings = [
            "The doctor examined the patient and documented their symptoms",
            "A treatment plan was discussed and prescribed",
            "Follow-up care was recommended"
        ]

    prompt = f"""<start_of_turn>user
You are a compassionate medical assistant helping patients understand their medical reports in simple language.

Medical Report:
{transcription[:600]}

Explain this report simply. Include summary, key findings, warnings, and specific questions.
<end_of_turn>
<start_of_turn>model
"""

    completion = f"""**Summary:** {description}

**Key Findings:**
1. {findings[0] if len(findings) > 0 else 'Doctor examined and assessed the patient'}
2. {findings[1] if len(findings) > 1 else 'Symptoms and medical history were reviewed'}
3. {findings[2] if len(findings) > 2 else 'A treatment plan was prescribed'}

**⚠️ Attention:** Follow up with your doctor if symptoms worsen or new symptoms appear.

**Questions for your doctor:**
1. {questions[0]}
2. {questions[1]}
3. {questions[2]}
<end_of_turn>"""

    return prompt, completion

In [9]:
prompts, completions = zip(*df.apply(
    lambda row: create_prompt_completion(
        row['transcription'],
        row['description'],
        row['medical_specialty']
    ), axis=1
))

df['prompt'] = list(prompts)
df['completion'] = list(completions)

print(f" Prompts created: {len(df)}")

 Prompts created: 4920


In [10]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_dataset = Dataset.from_pandas(train_df[['prompt', 'completion']])
print(f"Training examples: {len(train_dataset)}")

Training examples: 3936


## Step 4: Load MedGemma + LoRA

In [11]:
from kaggle_secrets import UserSecretsClient
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
import torch

token = UserSecretsClient().get_secret('HF_TOKEN')

# compresses model to fit in our gpu
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

MODEL_NAME="google/medgemma-1.5-4b-it"

print('Loading Tokenizer ...')
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME,token=token)

print('loading model ...')
model=AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    token=token
)

print('Medgemma loaded!!')

Loading Tokenizer ...


config.json:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

Medgemma loaded!!


In [12]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        'q_proj', 'k_proj',
        'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,}")
print(f"Total parameters: {total:,}")
print(f"We are training: {100 * trainable / total:.2f}% of the model")

Trainable parameters: 32,788,480
Total parameters: 2,523,011,440
We are training: 1.30% of the model


## Step 5: Load Fine-tuned Weights

In [13]:
from peft import PeftModel
model = PeftModel.from_pretrained(
    model,
    "gyxnova/medreport-explainer",
    token=token
)
model.eval()
print("Fine-tuned model loaded!")

adapter_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


adapter_model.safetensors:   0%|          | 0.00/131M [00:00<?, ?B/s]

Fine-tuned model loaded!


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:598: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.model.vision_tower.vision_model.encoder.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.vision_tower.vision_model.encoder.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.vision_tower.vision_model.encoder.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.vision_tower.vision_model.encoder.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.vision_tower.vision_model.encoder.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.vision_tower.vision_model.encoder.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.vision_tower.vision_model.encoder.layers.1.self_attn.k_proj.lora_A.default

## Step 6:Test Model

In [14]:
def generate_response(report_text):
    prompt = f"""<start_of_turn>user
You are a helpful medical assistant.
Explain this medical report in simple language.

Medical Report:
{report_text[:600]}
<end_of_turn>
<start_of_turn>model
**Summary:**"""

    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)
    return "**Summary:**" + response

# Test
sample = test_df.iloc[0]
response = generate_response(sample['transcription'])
print(response)

**Summary:**

This medical report is about a 50-year-old African American female who has end-stage renal disease (ESRD) and is on hemodialysis. She is getting a kidney transplant.

Here's a breakdown of the key points:

* **Patient:** A 50-year-old African American female.
* **Condition:** She has end-stage renal disease (ESRD), meaning her kidneys have stopped working properly.
* **Treatment:** She is currently receiving hemodialysis, which is a treatment to help her kidneys function temporarily.
* **Reason for Report:** The report is being written in preparation for a kidney transplant.
* **Transplant Plan:** The patient is eligible for a kidney transplant. This means a healthy kidney will be placed in her body to replace her failing one.
* **Preparation:** Before the transplant surgery, doctors will check her blood type and HLA typing (genetic markers) to make sure a compatible kidney can be found and used.
* **Decision:** Based on the patient's condition and the results of the bloo

## Step 7: Multilingual Gradio UI

In [15]:
!pip install sentencepiece gradio -q

import gradio as gr
import torch
import re
from transformers import MarianMTModel, MarianTokenizer

# Translation models
TRANSLATION_MODELS = {
    'hi': 'Helsinki-NLP/opus-mt-en-hi',
    'es': 'Helsinki-NLP/opus-mt-en-es',
    'ar': 'Helsinki-NLP/opus-mt-en-ar',
    'fr': 'Helsinki-NLP/opus-mt-en-fr',
    'bn': 'Helsinki-NLP/opus-mt-en-bn',
}

translation_cache = {}

# Preload all translation models upfront
print("Preloading all translation models...")
for lang, model_name in TRANSLATION_MODELS.items():
    try:
        print(f"Loading {lang}...")
        t_tok = MarianTokenizer.from_pretrained(model_name)
        t_mod = MarianMTModel.from_pretrained(model_name)
        translation_cache[lang] = (t_tok, t_mod)
        print(f"✅ {lang} ready!")
    except Exception as e:
        print(f"❌ {lang} failed: {e}")
print("All translation models loaded!")

def translate_text(text, target_lang):
    if target_lang == 'en':
        return text
    if target_lang not in TRANSLATION_MODELS:
        return text
    
    if target_lang not in translation_cache:
        print(f"Loading {target_lang} translation model...")
        model_name = TRANSLATION_MODELS[target_lang]
        t_tok = MarianTokenizer.from_pretrained(model_name)
        t_mod = MarianMTModel.from_pretrained(model_name)
        translation_cache[target_lang] = (t_tok, t_mod)
    
    t_tok, t_mod = translation_cache[target_lang]
    lines = text.split('\n')
    translated = []
    
    for line in lines:
        if not line.strip():
            translated.append(line)
            continue
        try:
            inputs = t_tok(line, return_tensors='pt',
                          truncation=True, max_length=512)
            outputs = t_mod.generate(**inputs)
            result = t_tok.decode(outputs[0], skip_special_tokens=True)
            translated.append(result)
        except Exception as e:
            translated.append(line)  # keep original if translation fails
    
    return '\n'.join(translated)

def generate_response(report_text):
    prompt = f"""<start_of_turn>user
You are a helpful medical assistant.
Explain this medical report in simple language.
You MUST end with exactly 3 questions the patient should ask their doctor.

Medical Report:
{report_text[:600]}
<end_of_turn>
<start_of_turn>model
**Summary:**"""

    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=400,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)
    
    if 'question' not in response.lower() and 'ask' not in response.lower():
        response += "\n\n**Questions to ask your doctor:**\n1. What do these findings mean for my long-term health?\n2. Do I need any follow-up tests or appointments?\n3. Are there any lifestyle changes I should make based on this report?"
    
    return "**Summary:**" + response

LANGUAGES = {
    'English 🇬🇧': 'en',
    'Hindi 🇮🇳': 'hi',
    'Spanish 🇪🇸': 'es',
    'Arabic 🇸🇦': 'ar',
    'French 🇫🇷': 'fr',
    'Bengali 🇧🇩': 'bn',
}

SAMPLES = [
    df['transcription'].iloc[0][:600],
    df['transcription'].iloc[10][:600],
    df['transcription'].iloc[50][:600],
]

def explain_with_status(report_text, language_choice):
    if not report_text.strip():
        yield "⚠️ Please paste your medical report.", ""
        return
    
    yield "⏳ Analyzing your report... (30-60 seconds)", ""
    response = generate_response(report_text)
    
    lang_code = LANGUAGES[language_choice]
    if lang_code != 'en':
        yield f"🌍 Translating to {language_choice}...", ""
        response = translate_text(response, lang_code)
    
    yield "✅ Done!", response

try:
    demo.close()
except:
    pass

with gr.Blocks(title="MedReport Explain", theme=gr.themes.Soft()) as demo:
    gr.HTML("""
    <div style='text-align:center; padding:1rem;'>
        <h1>🏥 MedReport Explain</h1>
        <p>Understand your medical report in your language.<br>
        Powered by <strong>MedGemma</strong> · Built for underserved patients worldwide.</p>
        <p>🇮🇳 Hindi · 🇪🇸 Spanish · 🇧🇩 Bengali · 🇸🇦 Arabic · 🇫🇷 French</p>
    </div>
    """)

    with gr.Row():
        with gr.Column():
            report_input = gr.Textbox(
                label="📄 Paste Your Medical Report",
                placeholder="Paste your lab report, discharge summary, or doctor notes here...",
                lines=12
            )
            language_input = gr.Dropdown(
                choices=list(LANGUAGES.keys()),
                value='English 🇬🇧',
                label="🌍 Select Language"
            )
            submit_btn = gr.Button("🔍 Explain My Report", variant="primary", size="lg")
            gr.Examples(
                examples=[[s, 'English 🇬🇧'] for s in SAMPLES],
                inputs=[report_input, language_input],
                label="📋 Try a Sample Report"
            )

        with gr.Column():
            status = gr.Markdown(value="")
            output = gr.Markdown(value="*Your explanation will appear here...*")
            gr.HTML("""
            <div style='padding:0.75rem; background:#fff3cd;
                        border-radius:8px; font-size:0.85em; color:#856404;'>
            ⚠️ Always consult your doctor before making medical decisions.
            </div>
            """)

    submit_btn.click(
        fn=explain_with_status,
        inputs=[report_input, language_input],
        outputs=[status, output]
    )

demo.launch(share=True, debug=False)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 67.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
langchain-core 0.3.79 requires packaging<26.0.0,>=23.2.0, but you have packaging 26.0rc2 which is incompatible.
fastai 2.8.4 requires fastcore<1.9,>=1.8.0, but you have fastcore 1.11.3 which is incompatible.
Preloading all translation models...
Loading hi...


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

...

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

✅ fr ready!
Loading bn...
❌ bn failed: Helsinki-NLP/opus-mt-en-bn is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`
All translation models loaded!
* Running on local URL:  http://127.0.0.1:7860


model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

* Running on public URL: https://dfac0f958b9ed9b2fa.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
